##  Implementing Custom Unet Training

In [117]:
!git clone https://github.com/VikramShenoy97/Human-Segmentation-Dataset

fatal: destination path 'Human-Segmentation-Dataset' already exists and is not an empty directory.


In [118]:
import os
import time
import torch

from PIL import Image

from torch import nn
from torchvision import transforms


from torch.utils.data import DataLoader , Dataset
from torch.optim import Adam,AdamW , SGD

In [119]:
class SeqmentationDataset(Dataset):
  def __init__(self,img_dir,mask_dir,transform=None):
    self.image_dir=img_dir
    self.mask_dir=mask_dir
    self.transform=transforms.Compose([
        transforms.Resize((512,512)),
        transforms.ToTensor()
    ])

    valid_extension={".jpg",".jpeg",".png"}
    self.images=[img_file for img_file in os.listdir(img_dir) if os.path.splitext(img_file)[1] in valid_extension]


  def __len__(self):
    return len(self.images)

  def __getitem__(self, index):
    image_path=os.path.join(self.image_dir,self.images[index])
    name,ext=os.path.splitext(self.images[index])
    mask_path=os.path.join(self.mask_dir,f"{name}.png")


    image=Image.open(image_path).convert("RGB")
    mask=Image.open(mask_path).convert("L")

    image=self.transform(image)
    mask=self.transform(mask)

    mask=(mask>0.5).float()

    return image,mask


In [120]:
def get_dataloader(image_dir,mask_dir,batch_size=2,shuffle=True):
  dataset=SeqmentationDataset(image_dir,mask_dir)
  return DataLoader(dataset,batch_size=batch_size,shuffle=shuffle)

# UNET

![](https://drive.google.com/uc?id=1-VqrLBuLuzcIxk3QNTIHP5u1JW93NNy5)

In [121]:
class DoubleConv(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.conv_op=nn.Sequential(
        nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU(inplace=True)

    )

    def forward(self,x):
      return self.conv_op(x)

In [122]:
class DownSample(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.conv=DoubleConv(in_channels, out_channels)
    self.pool=nn.MaxPool2d(kernel_size=2,stride=2)
  def forward(self, x):
    down=self.conv(x)
    p=self.pool(down)
    return down,p

In [123]:
class Upsample(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.up=nn.ConvTranspose2d(in_channels,in_channels//2,kernel_size=2,stride=2)
    self.conv=DoubleConv(in_channels,out_channels)

  def forward(self,x1,x2):
    x1=self.up(x1)
    x=torch.cat([x1,x2],1)
    return self.conv(x)



In [124]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_op = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv_op(x)


class Unet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        # Encoder
        self.down1 = DownSample(in_channels, 64)
        self.down2 = DownSample(64, 128)
        self.down3 = DownSample(128, 256)
        self.down4 = DownSample(256, 512)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up1 = Upsample(1024, 512)
        self.up2 = Upsample(512, 256)
        self.up3 = Upsample(256, 128)
        self.up4 = Upsample(128, 64)

        # Output
        self.out = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        d1, p1 = self.down1(x)
        d2, p2 = self.down2(p1)
        d3, p3 = self.down3(p2)
        d4, p4 = self.down4(p3)

        # Bottleneck
        bottleneck = self.bottleneck(p4)

        # Decoder with skip connections
        u1 = self.up1(bottleneck, d4)
        u2 = self.up2(u1, d3)
        u3 = self.up3(u2, d2)
        u4 = self.up4(u3, d1)

        return self.out(u4)


In [125]:
# class Unet(nn.Module):
#   def __init__(self,in_channels,num_classes):
#     super().__init__()
#     self.down_convolution_1=DownSample(in_channels,64)
#     self.down_convolution_2=DownSample(64,128)
#     self.down_convolution_3=DownSample(128,256)
#     self.down_convolution_4=DownSample(256,512)


#     self.bottle_neck=DoubleConv(512,1024)


#     self.down_convolution_1=Upsample(1024,512)
#     self.down_convolution_2=Upsample(512,256)
#     self.down_convolution_3=Upsample(256,128)
#     self.down_convolution_4=Upsample(128,64)

#     self.out=nn.Conv2d(in_channels=64,out_channels=num_classes,kernel_size=1)

#   def forward(self,x):
#     down_1,p1=self.down_convolution_1(x)
#     down_2,p2=self.down_convolution_1(p1)
#     down_3,p3=self.down_convolution_1(p2)
#     down_4,p4=self.down_convolution_1(p3)

#     bottle_neck=self.bottle_neck(p4)

#     up_1=self.up_convolution_1(bottle_neck,down_4)
#     up_2=self.up_convolution_2(up_1,down_3)
#     up_3=self.up_convolution_3(up_2,down_2)
#     up_4=self.up_convolution_4(up_3,down_1)

#     out=self.out(up_4)

#     return out





In [126]:
class DiceLoss(nn.Module):
  def __init__(self, smooth=1e-6):
    super(DiceLoss, self).__init__()
    self.smooth=smooth

  def forward(self,inputs,targets):
    inputs=inputs.view(-1)
    targets=targets.view(-1)


    intersection=(inputs*targets).sum()
    dice_score=(2.*intersection+self.smooth)/(inputs.sum()+targets.sum()+self.smooth)

    return 1-dice_score


In [127]:
class BCEWithDiceLoss(nn.Module):
  def __init__(self, smooth=1e-6):
    super(BCEWithDiceLoss, self).__init__()

    self.bce=nn.BCEWithLogitsLoss()
    self.dice=DiceLoss()

  def forward(self,inputs,targets):
    bce_loss=self.bce(inputs,targets)
    dice_loss=self.dice(inputs,targets)

    return 0.5*bce_loss+dice_loss


In [128]:
## Training Loop
def train(model,dataloader,epochs=2,lr=0.001,save_path="unet_model",load_path=None):

  device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model=model.to(device)

  if load_path and os.path.exists(load_path):
    print(f"Model loaded from {load_path}")
    model.load_state_dict(torch.load(load_path,map_location=device))
  else:
    print(f"No Check Point Found,training from scratch")


  print(device)
  model.to(device)

  criterion=BCEWithDiceLoss()
  optimizer=Adam(model.parameters(),lr=lr)

  for epoch in range(epochs):
    model.train()
    epoch_loss=0
    for images,masks in dataloader:
      images,masks=images.to(device),masks.to(device)
      optimizer.zero_grad()

      outputs=model(images)

      loss=criterion(outputs,masks)
      loss.backward()
      optimizer.step()

      epoch_loss+=loss.item()
    avg_loss=epoch_loss/len(dataloader)

    print(f"Epoch {epoch+1}/{epochs} Loss: {avg_loss:.4f} , LR:{lr}")

    if epoch%10==0 and epoch>0:
      torch.save(model.state_dict(),f"{save_path}.pth")


  torch.save(model.state_dict(),f"{save_path}_final.pth")
  print(f"Model saved to {save_path}_final.pth")







In [129]:
dataloader=get_dataloader("/content/Human-Segmentation-Dataset/Training_Images","/content/Human-Segmentation-Dataset/Ground_Truth",batch_size=8,shuffle=True)

In [130]:
model=Unet(in_channels=3,num_classes=1)

In [ ]:
train(model,dataloader,epochs=20,lr=0.001)

No Check Point Found,training from scratch
cuda
Epoch 1/20 Loss: 1.0427 , LR:0.001
Epoch 2/20 Loss: 0.5240 , LR:0.001
Epoch 3/20 Loss: 1.0266 , LR:0.001
Epoch 4/20 Loss: 0.8776 , LR:0.001
Epoch 5/20 Loss: 0.7306 , LR:0.001
Epoch 6/20 Loss: 2.3839 , LR:0.001
Epoch 7/20 Loss: 1.2983 , LR:0.001
Epoch 8/20 Loss: 1.0271 , LR:0.001
Epoch 9/20 Loss: 0.9053 , LR:0.001
Epoch 10/20 Loss: 0.7992 , LR:0.001
Epoch 11/20 Loss: 0.6440 , LR:0.001
Epoch 12/20 Loss: 1.3384 , LR:0.001
Epoch 13/20 Loss: 0.7289 , LR:0.001
Epoch 14/20 Loss: 4.9568 , LR:0.001
Epoch 15/20 Loss: 37.0035 , LR:0.001
Epoch 16/20 Loss: 0.9970 , LR:0.001
Epoch 17/20 Loss: 81666.9745 , LR:0.001
Epoch 18/20 Loss: 47.5546 , LR:0.001


## Inferencing on train model

In [ ]:
import numpy as np

# Load model and predict with stats
def predict(model_path, input_image_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load model
    model = Unet(in_channels=3, num_classes=1)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # Track start time
    total_start_time = time.time()

    # Image preprocessing
    preprocess_start_time = time.time()
    image = Image.open(input_image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)
    preprocess_end_time = time.time()

    # Model inference
    inference_start_time = time.time()
    with torch.no_grad():
        output = model(image_tensor)
        output = torch.sigmoid(output)
    inference_end_time = time.time()

    # Postprocessing
    postprocess_start_time = time.time()
    mask = output.squeeze(0).squeeze(0).cpu().numpy()
    mask = (mask > 0.4).astype(np.uint8) * 255
    mask_image = Image.fromarray(mask)

    combined = Image.new("RGB", (512 * 2, 512))
    combined.paste(image.resize((512, 512)), (0, 0))
    combined.paste(mask_image.convert("RGB"), (512, 0))
    combined.save("output.jpg")
    postprocess_end_time = time.time()

    # Calculate timing stats
    total_end_time = time.time()

    preprocess_time = preprocess_end_time - preprocess_start_time
    inference_time = inference_end_time - inference_start_time
    postprocess_time = postprocess_end_time - postprocess_start_time
    total_time = total_end_time - total_start_time

    # Print stats
    print("\nPrediction completed! Stats:")
    print(f"  Image Preprocessing Time: {preprocess_time:.4f} seconds")
    print(f"  Model Inference Time: {inference_time:.4f} seconds")
    print(f"  Postprocessing Time: {postprocess_time:.4f} seconds")
    print(f"  Total Prediction Time: {total_time:.4f} seconds")
    print("Prediction saved as output.jpg")


In [ ]:
predict(model_path="/content/unet_model_final.pth", input_image_path="/content/Human-Segmentation-Dataset/Training_Images/109.jpg")